# PyTorch
In this exercise, we will look at some basic functionality of PyTorch. Your are free to use other DL frameworks for your exercises and your project. However, the master solutions and code examples will be in PyTorch.

The [PyTorch documentation](https://pytorch.org/docs/stable/index.html) offers information on its functionality. A lot of the time, your specific question will also have been asked on the [PyTorch Forum](https://discuss.pytorch.org/), often with competent answers by the core developers (Google will find the relevant thread for you).

First, we have to install PyTorch. We will install the basic version for this exercise. For your project, if you want to run on a GPU, you'll have to make sure to have a PyTorch version installed that is compatible with the CUDA version of your NVIDIA drivers. PyTorch has an [installation guide](https://pytorch.org/get-started/locally/) that will help you with getting the right version.

In [1]:
%pip install -q torch ipywidgets ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

## Tensor operations
Most of PyTorch's operations have the same name as in NumPy. The basic object for storing data is the `torch.tensor`, the equivalent of the `np.array`. With the help of the [Tensor tutorial](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html), do the following:

- Create a `torch.tensor` with the elements `[[1, 2], [3, 4]]`
- Create a tensor of ones/zeros with the same shape and dtype
- Create a random tensor of the same shape
- Print the tensor's shape, data type and device
- Try to move it to the GPU
- For Mac users: Try to move it to [MPS](https://pytorch.org/docs/stable/notes/mps.html)
- Check out indexing/slicing operations, and how you can assign values to a slice.
- Combine tensors with `torch.cat` and `torch.stack`. What are the differences?
- Multiply tensors, element-wise and with matrix multiplication.

In [3]:
# Tensor creation
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)

# Initialize tensor with same shape as x_data
x_ones = torch.ones_like(x_data)
x_zeros = torch.zeros_like(x_data)
try:
    x_rand = torch.rand_like(x_data)
except RuntimeError as err:
    print('Error:', err)
x_rand = torch.rand_like(x_data, dtype=torch.float) # have to override the data type
tensor = torch.rand(3, 4)

# Tensor properties
print(f"Shape of tensor: {tensor.shape}")
print(f"Data type of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

# CUDA
print(f"GPU is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    tensor = tensor.to('cuda')
    print(f"Device tensor is stored on: {tensor.device}")

# MPS
print(f"MPS is available: {torch.backends.mps.is_available()}")
if torch.backends.mps.is_available():
    tensor = tensor.to('mps')
    print(f"Device tensor is stored on: {tensor.device}")

# Side note: If we move a tensor to the GPU/MPS, we have to reassign it. If we move an nn.Module, it does this for us.
# Explained in: https://discuss.pytorch.org/t/moving-tensor-to-cuda/39318/2

# Indexing, slicing
tensor = torch.eye(4, 4)
print(tensor[:2, 1:3])
tensor[:,1] = 2
print(tensor)

# Cat vs. stack
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print('torch.cat:', t1.shape)
t2 = torch.stack([tensor, tensor], dim=2)
print('torch.stack:', t2.shape)

# Element-wise multiplication
tensor = torch.randn(2, 3)
res = tensor.mul(tensor)
print(res.shape)
assert torch.all(tensor * tensor == tensor.mul(tensor))

# Matrix multiplication
res = tensor.matmul(tensor.T)
print(res.shape)
assert torch.all(tensor @ tensor.T == tensor.matmul(tensor.T))

Error: "check_uniform_bounds" not implemented for 'Long'
Shape of tensor: torch.Size([3, 4])
Data type of tensor: torch.float32
Device tensor is stored on: cpu
GPU is available: False
MPS is available: True
Device tensor is stored on: mps:0
tensor([[0., 0.],
        [1., 0.]])
tensor([[1., 2., 0., 0.],
        [0., 2., 0., 0.],
        [0., 2., 1., 0.],
        [0., 2., 0., 1.]])
torch.cat: torch.Size([4, 12])
torch.stack: torch.Size([4, 4, 2])
torch.Size([2, 3])
torch.Size([2, 2])


## Neural Network Basics
Solve the followings tasks with the help of the [Neural networks tutorial](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html).

The `nn.Module` is the basic class for layers, networks and models. All parameters of an `nn.Module` are automatically discovered by PyTorch and updated by back-propagation.

First, define a neural network (as a subclass of `nn.Module`) with two linear layers and a ReLU non-linearity in between. Make the input, output, and inner dimensions parameters of your network.

In [4]:
import torch.nn as nn

In [5]:
class MyNetwork(nn.Module):
    
    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

dim_in = 40
dim_out = 20
net = MyNetwork(dim_in, dim_out)
print(net)

MyNetwork(
  (linear1): Linear(in_features=40, out_features=10, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=10, out_features=20, bias=True)
)


Move the entire network to the GPU/MPS.

In [6]:
print(f"Device before: {net.linear1.weight.device}")
if torch.backends.mps.is_available():
    net.to("mps")  # does not have to be reassigned
print(f"Device after: {net.linear1.weight.device}")

Device before: cpu
Device after: mps:0


Print the parameters of your network.

In [7]:
params = list(net.parameters())
print(len(params))
for name, params in net.named_parameters():
    print(name, params.shape)

4
linear1.weight torch.Size([10, 40])
linear1.bias torch.Size([10])
linear2.weight torch.Size([20, 10])
linear2.bias torch.Size([20])


Run a single forward-pass with a random input.

In [8]:
batch_size = 5
x = torch.randn(batch_size, dim_in)
try:
    output = net(x)
except RuntimeError as err:
    print("Error:", err)
if torch.backends.mps.is_available():
    x = x.to('mps')
    output = net(x)
print(output.shape)
print(output)

Error: Tensor for argument input is on cpu but expected on mps
torch.Size([5, 20])
tensor([[-0.4602, -0.2159,  0.1484,  0.3001, -0.3524,  0.0168,  0.2795, -0.1006,
         -0.3213, -0.2243, -0.1462,  0.1528, -0.5161,  0.2478,  0.0746,  0.0203,
          0.2099, -0.2641,  0.5015,  0.2235],
        [-0.4164, -0.2970,  0.3894, -0.2273, -0.0905, -0.1101,  0.2136, -0.1591,
         -0.2877,  0.3087,  0.4465, -0.1293, -0.3185,  0.0427, -0.0754,  0.4639,
          0.5504, -0.2424,  0.2558, -0.0585],
        [ 0.0968, -0.0881,  0.1934,  0.0683, -0.0700,  0.2239,  0.5255, -0.5334,
         -0.4271, -0.0958,  0.3300,  0.5404, -0.6061, -0.0362, -0.4884,  0.4885,
          0.2827, -0.3373,  0.0155, -0.3189],
        [-0.3673, -0.3132,  0.5735, -0.2037,  0.3386,  0.7374,  0.2665, -0.3485,
          0.3893, -0.2533,  0.1081,  0.4263, -0.0429, -0.1081, -0.3078,  0.9160,
         -0.0173, -0.8272,  0.9543, -0.7630],
        [-0.8831, -0.5744,  0.9483, -0.4833,  0.0511,  0.1307, -0.1384, -0.2146,
    

Define a `nn.MSELoss` and a random target.

In [9]:
criterion = nn.MSELoss()
target = torch.randn(batch_size, dim_out, device='mps')

Compute the loss and run backpropagation.

In [10]:
loss = criterion(output, target)
print(loss)
"""
Don't forget to call .zero_grad before calling .backward!
Otherwise gradients will accumulate over multiple steps,
and so will the computation graph that has to be kept in memory.
=> You'll get an out-of-memory error.
"""
net.zero_grad()
loss.backward()

tensor(0.8748, device='mps:0', grad_fn=<MseLossBackward0>)


Update the parameters of your network with a learning rate of 0.01.

In [11]:
learning_rate = 0.01
for param in net.parameters():
    param.data = param.data - param.grad.data * learning_rate

Use the `AdamOptimizer` instead to update your parameters.

In [12]:
from torch.optim import Adam

optimizer = Adam(net.parameters(), lr=0.01)

output = net(x)
loss = criterion(output, target)

optimizer.zero_grad()  # now we call zero_grad on the optimizer instead of the net
loss.backward()
optimizer.step()  # optimizer.step updates the model's parameters